## RISE baseline (Randomized Input Sampling for Explanation)
 **vanilla RISE** baseline for generating saliency maps using *random binary masks* and measuring how masking affects the face recognition model’s similarity / identification score. This provides a strong, model-agnostic reference point before introducing our contrastive variant.

Reference implementation and paper link: https://github.com/eclique/RISE

In [ ]:
import os, json, time, random
import numpy as np
import cv2

# ---- dataset/cache paths ----
SPLIT_PATH = "splits/lfw_1N_split.json"
G_IDS_PATH = "cache/gallery_ids.npy"
G_EMB_PATH = "cache/G.npy"

# ---- experiment control ----
MASTER_SEED = 123
K_EXPERIMENTS = 1680
MAX_PROBES_PER_ID = 1

# ---- RISE hyperparams ----
N = 1000
s = 8
p1 = 0.5
BATCH_SAVE = 50

# ---- output ----
OUT_DIR = "results/rise_alignedchip_baseline_multi"
os.makedirs(OUT_DIR, exist_ok=True)

print("MASTER_SEED:", MASTER_SEED)
print("K_EXPERIMENTS:", K_EXPERIMENTS)
print("N,s,p1:", N, s, p1)
print("OUT_DIR:", OUT_DIR)

with open(SPLIT_PATH) as f:
    split = json.load(f)

gallery = split["gallery"]
probes  = split["probes"]

gallery_ids = np.load(G_IDS_PATH, allow_pickle=True).tolist()
G = np.load(G_EMB_PATH).astype(np.float32)

id_to_index = {pid: i for i, pid in enumerate(gallery_ids)}

print("Loaded gallery embeddings:", G.shape)
print("Loaded split identities:", len(gallery))

In [2]:
import onnxruntime as ort
from insightface.app import FaceAnalysis

print("onnxruntime version:", ort.__version__)
print("available providers:", ort.get_available_providers())

app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
)
app.prepare(ctx_id=0, det_size=(640, 640))
print("ArcFace ready.")

rec = app.models["recognition"]  # ArcFaceONNX

def get_embedding_from_chip(chip_bgr: np.ndarray) -> np.ndarray:
    """
    chip_bgr: (112,112,3) BGR aligned face crop
    returns: (512,) float32 L2-normalized
    """
    assert chip_bgr.shape == (112,112,3), f"Expected (112,112,3), got {chip_bgr.shape}"
    feat = rec.get_feat(chip_bgr)
    feat = np.asarray(feat).reshape(-1).astype(np.float32)
    n = float(np.linalg.norm(feat) + 1e-12)
    return feat / n

onnxruntime version: 1.24.2
available providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /home/stu10/s7/lpr5476/.insightface/models/buffalo_l/1k3d68.onnx landm

In [ ]:
import importlib
import rise
importlib.reload(rise)

from rise import (
    generate_mask_rise,
    apply_mask_black,
    build_aligned_chip_112,
    get_embedding_from_chip,
    run_rise,
    save_overlay_images,
    cosine_weight,
)

print("rise.py loaded.")

In [ ]:
import pandas as pd

valid_ids = [
    pid for pid in probes
    if pid in gallery and pid in id_to_index and len(probes[pid]) > 0
]
valid_ids = sorted(valid_ids)

sel_rng = random.Random(MASTER_SEED)
K = min(K_EXPERIMENTS, len(valid_ids))
picked_ids = sel_rng.sample(valid_ids, K)

results = []
for exp_i, pid in enumerate(picked_ids):
    probe_list = list(probes[pid])
    sel_rng.shuffle(probe_list)
    probe_list = probe_list[:max(1, MAX_PROBES_PER_ID)]

    for k, probe_path in enumerate(probe_list):
        run_seed = MASTER_SEED * 10_000 + exp_i * 100 + k

        try:
            out = run_rise(
                true_id=pid,
                probe_path=probe_path,
                G=G,
                id_to_index=id_to_index,
                rec=rec,
                app=app,
                weight_fn=cosine_weight,
                run_seed=run_seed,
                out_dir=OUT_DIR,
                N=N, s=s, p1=p1,
                batch_save=BATCH_SAVE,
            )
            results.append({
                "true_id":      out["true_id"],
                "probe_file":   os.path.basename(out["probe_path"]),
                "probe_path":   out["probe_path"],
                "run_seed":     out["run_seed"],
                "failures":     out["failures"],
                "w_clean":      out["w_clean"],
                "w_black":      out["w_black"],
                "saliency_path": out["saliency_path"],
            })
        except Exception as e:
            results.append({
                "true_id":      pid,
                "probe_file":   os.path.basename(probe_path),
                "probe_path":   probe_path,
                "run_seed":     run_seed,
                "failures":     None,
                "w_clean":      None,
                "w_black":      None,
                "saliency_path": None,
                "error":        repr(e),
            })
            print("[error]", pid, probe_path, "->", repr(e))

df = pd.DataFrame(results)
df

In [ ]:
import matplotlib.pyplot as plt

FIG_DIR = os.path.join(OUT_DIR, "figures")

for _, row in df.iterrows():
    sal_path = row.get("saliency_path")
    if not isinstance(sal_path, str) or not os.path.exists(sal_path):
        continue

    img = cv2.imread(row["probe_path"])
    chip_bgr = build_aligned_chip_112(img, app)
    chip_rgb = cv2.cvtColor(chip_bgr, cv2.COLOR_BGR2RGB)
    sal = np.load(sal_path).astype(np.float32)

    plt.figure(figsize=(4, 4))
    plt.imshow(chip_rgb)
    plt.imshow(sal, alpha=0.50, vmin=0, vmax=1)
    plt.axis("off")
    plt.title(f"{row['true_id']} | {row['probe_file']}\n"
              f"w_clean={row['w_clean']:.3f} w_black={row['w_black']:.3f}")
    plt.show()

In [ ]:
saved = []
for _, row in df.iterrows():
    sal_path = row.get("saliency_path")
    if not isinstance(sal_path, str) or not os.path.exists(sal_path):
        print(f"Skip {row['true_id']}: no saliency saved")
        continue

    try:
        chip_out, heat_out, overlay_out = save_overlay_images(
            true_id=row["true_id"],
            probe_path=row["probe_path"],
            sal_path=sal_path,
            w_clean=float(row["w_clean"]),
            w_black=float(row["w_black"]),
            fig_dir=FIG_DIR,
            N=N, s=s, p1=p1,
            app=app,
        )
        saved.append({
            "true_id":      row["true_id"],
            "probe_file":   row["probe_file"],
            "chip_png":     chip_out,
            "saliency_png": heat_out,
            "overlay_png":  overlay_out,
        })
        print("Saved:", overlay_out)
    except Exception as e:
        print("[save error]", row["true_id"], row["probe_path"], "->", repr(e))

pd.DataFrame(saved)

In [ ]:
# Save run summary CSV
import pandas as pd

summary_path = os.path.join(
    OUT_DIR,
    f"summary_K{K_EXPERIMENTS}_N{N}_s{s}_p{p1}_MASTERSEED{MASTER_SEED}.csv"
)
df.to_csv(summary_path, index=False)
print("Saved:", summary_path)

ok = df["saliency_path"].notna().sum()
print(f"Completed: {ok} / {len(df)}")
if "error" in df.columns:
    print(f"Errors: {df['error'].notna().sum()}")